Creditos:

Para la detección de vehiculos con yolo se tomó como ayuda el siguiente notebook de kaggle


https://www.kaggle.com/code/dharmaketre/vehicle-counter-object-detection-using-yolo

Instalación de librerias necesarias

In [1]:
!pip install ultralytics opencv-python

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 31.9 MB/s  0:00:00
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ------------- -------------------------- 13.1/40.2 MB 61.3 MB/s eta 0:00:01
   ------------------------ --------------- 24.9/40.2 MB 60.3 MB/s eta 0:00:01
   ------------------------------------ --- 36.7/40.2 MB 59.6 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 55.0 MB/s  0:00:00
   ---------------------------------------- 0.0/828.7 kB ? eta -:--:--
   ---------------------------------------- 828.7/828.7 kB 35.4 MB/s  0:00:00
   ---------------------------------------- 0.0/51.8 MB ? eta -:--:--
   ---------- ----------------------------- 13.4/51.8 MB 61.6 MB/s eta 0:00:01
   ------------------- -------------------- 25.2/51.8 MB 60.4 MB/s eta 0:00:01
   ----------------------------- ---------- 37.7/51.8 MB 60.8 MB/s eta 0:00:01
   ---------------

In [8]:
!pip install plotly ipywidgets

In [10]:
!pip install anywidget ipywidgets plotly


   ---------------------------------------- 0/2 [psygnal]
   ---------------------------------------- 2/2 [anywidget]



Imports necesarios

In [1]:
import cv2
import plotly.graph_objects as go
from ultralytics import YOLO
from IPython.display import display # Usado para mostrar plotly en Jupyter

Ejecución principal, se obtiene cada frame de un video de prueba usando OpenCV, luego con YoloV8 se detectan los vehículos en el frame para extraer sus bounding boxes, además, si estas bounding boxes estan cerca de una linea de conteo dibujada en pantalla se agrega el vehículo a un arreglo de vehívulos contados; las métricas de vehículos contados vs tiempo (frames) se pueden visualizar en tiempo real con una grafica de plotly

In [8]:
# Configuración de YOLO y Video
VIDEO_PATH = "../media/YoloV8 test.mp4"
model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(VIDEO_PATH)

# Variables de estado
already_counted = set() # Vehículos ya contados para evitar duplicados
vehiculos_contados = [] # Lista para almacenar IDs de vehículos contados
history_x, history_y = [0], [0] # Para almacenar el historial de conteo para la gráfica
frame_idx = 0

# Configuración de la Gráfica de Plotly
fig = go.FigureWidget()
fig.add_scatter(x=history_x, y=history_y, mode='lines+markers', name='Vehículos', line=dict(color='firebrick'))
fig.update_layout(
    title="Conteo de Vehículos en Tiempo Real (Plotly)",
    xaxis_title="Frames",
    yaxis_title="Total Detectado",
    template="plotly_dark"
)

# Mostrar la gráfica
display(fig)

while cap.isOpened():
    success, frame = cap.read() # Obtener el siguiente frame del video
    if not success: break

    frame_idx += 1
    frame = cv2.resize(frame, (None, None), fx=0.5, fy=0.5) # Reducir resolución para mejorar rendimiento de YOLO
    h, w, _ = frame.shape
    line_y = int(h * 0.6) # Posición de la línea de conteo (60% de la altura del frame)

    # Detección de YOLO, solo va a detectar vehículos (clases 2, 3, 5, 7 corresponden a carros, motos, buses y camiones)
    results = model.track(frame, persist=True, classes=[2, 3, 5, 7], verbose=False)

    # Extraemos los IDs y las coordenadas de las cajas para el conteo
    if results[0].boxes.id is not None:
        ids = results[0].boxes.id.cpu().numpy().astype(int)
        boxes = results[0].boxes.xyxy.cpu().numpy()

        for box, obj_id in zip(boxes, ids):
            x1, y1, x2, y2 = map(int, box)
            cy = int((box[1] + box[3]) / 2)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2) # Dibujar las bounding boxes del vehículo detectado
            
            # Lógica de conteo
            if obj_id not in already_counted and abs(cy - line_y) < 15: # Si el centro del vehículo está cerca de la línea de conteo y si no ha sido contado antes

                # Agregar el ID del vehículo al set y a la lista de contados
                already_counted.add(obj_id)
                vehiculos_contados.append(obj_id)
                
                # Actualizar los datos del conteo
                history_x.append(frame_idx)
                history_y.append(len(already_counted))
                
                # Actualización del gráfico de Plotly
                with fig.batch_update():
                    fig.data[0].x = history_x
                    fig.data[0].y = history_y

    # UI de Video
    cv2.line(frame, (0, line_y), (w, line_y), (0, 255, 255), 2) # Linea horizontal para conteo
    cv2.putText(frame, f"Total: {len(already_counted)}", (20, 40), 1, 2, (0, 255, 0), 2) # Conteo total en pantalla
    cv2.imshow("Video Tracking", frame)


    # Salir de la aplicación al presionar 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

FigureWidget({
    'data': [{'line': {'color': 'firebrick'},
              'mode': 'lines+markers',
              'name': 'Vehículos',
              'type': 'scatter',
              'uid': 'aec89b5a-1e4a-4026-aafb-0d6fb2e5a810',
              'x': [0],
              'y': [0]}],
    'layout': {'template': '...',
               'title': {'text': 'Conteo de Vehículos en Tiempo Real (Plotly)'},
               'xaxis': {'title': {'text': 'Frames'}},
               'yaxis': {'title': {'text': 'Total Detectado'}}}
})